# QVAC Python SDK — notebook facade

`tetherto.qvac_sdk.notebook.SyncClient` is the synchronous, data-science-native client for Jupyter and REPLs: it runs the async client on a background event-loop thread, so every call is a plain blocking call (no `await`), returns numpy/pandas-native data, and streams completions live into the cell.

Install the `notebook` extra:

```bash
pip install "tetherto-qvac-sdk[notebook,bare-rpc]"
```

A worker must be available (see the docs' "Install the worker" step); against a source checkout, set `QVAC_SDK_DIR` or pass `SyncClient(sdk_dir=...)`.

In [ ]:
from tetherto.qvac_sdk.models import EMBEDDINGGEMMA_300M_Q4_0, QWEN3_600M_INST_Q4
from tetherto.qvac_sdk.notebook import SyncClient

# One client for the whole notebook; a daemon thread runs the event loop.
client = SyncClient()

## Embeddings as numpy arrays

In [ ]:
embed_model = client.load_model(model_src=EMBEDDINGGEMMA_300M_Q4_0)

vector = client.embed(embed_model, "hello from the notebook facade")
print("one text ->", type(vector).__name__, vector.shape, vector.dtype)

matrix = client.embed(embed_model, ["cats and dogs", "kittens and puppies"])
print("a batch  ->", matrix.shape)

## Batch embeddings as a pandas DataFrame

`embed_frame` returns a DataFrame indexed by the input text, one column per dimension — the last expression renders as a table in the cell.

In [ ]:
frame = client.embed_frame(
    embed_model,
    ["quantum computing", "espresso machine", "qubit entanglement"],
)
client.unload_model(embed_model)
frame.iloc[:, :5]  # first 5 dims

## Completion, streaming live into the cell

`completion(..., live=True)` updates the output area as tokens arrive (IPython display in a notebook, incremental stdout elsewhere) and returns the full text.

In [ ]:
llm = client.load_model(model_src=QWEN3_600M_INST_Q4, model_config={"n_ctx": 2048})
text = client.completion(
    llm,
    "Explain what an embedding is in one sentence.",
    predict=256,
    temp=0,
    seed=42,
)
client.unload_model(llm)
print(f"\n(returned {len(text)} chars)")

In [ ]:
client.close()